# Labo III
# Multinacional - Prediccion de Ventas

## Importamos librerias

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer


# funciones

In [5]:
# =============================================================================
# PARTE 2: FUNCIONES UTILITARIAS
# =============================================================================

def display_dataframe_info(df, name="DataFrame"):
    """
    Muestra información detallada sobre un DataFrame
    """
    print(f"\n{'='*50}")
    print(f"📊 INFORMACIÓN DE {name.upper()}")
    print(f"{'='*50}")
    print(f"🔹 Forma: {df.shape}")
    print(f"🔹 Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"🔹 Rango de fechas: {df['periodo'].min()} a {df['periodo'].max()}" if 'periodo' in df.columns else "")
    print(f"\n📋 Tipos de datos:")
    print(df.dtypes.value_counts())
    print(f"\n🔍 Valores nulos:")
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("✅ No hay valores nulos")
    
    if 'tn' in df.columns:
        print(f"\n📈 Estadísticas de 'tn':")
        print(f"   • Total: {df['tn'].sum():.2f}")
        print(f"   • Promedio: {df['tn'].mean():.4f}")
        print(f"   • Mediana: {df['tn'].median():.4f}")
        print(f"   • Ceros: {(df['tn'] == 0).sum():,} ({(df['tn'] == 0).mean()*100:.1f}%)")

def validate_date_format(df, date_col='periodo'):
    """
    Valida y convierte formato de fecha
    """
    print(f"🕐 Validando formato de fecha en columna '{date_col}'...")
    
    if df[date_col].dtype == 'object':
        # Intentar convertir formato YYYYMM a datetime
        try:
            df[date_col] = pd.to_datetime(df[date_col], format='%Y%m')
            print("✅ Fecha convertida desde formato YYYYMM")
        except:
            try:
                df[date_col] = pd.to_datetime(df[date_col])
                print("✅ Fecha convertida con auto-detección")
            except:
                print("❌ Error al convertir fechas")
                return False
    
    print(f"📅 Rango de fechas: {df[date_col].min()} a {df[date_col].max()}")
    return True

def get_product_customer_lifecycles(sales_df):
    """
    Calcula los ciclos de vida de productos y clientes
    """
    print("🔍 Calculando ciclos de vida de productos y clientes...")
    
    # Ciclo de vida de productos
    product_lifecycle = sales_df.groupby('product_id')['periodo'].agg(['min', 'max']).reset_index()
    product_lifecycle.columns = ['product_id', 'product_first_sale', 'product_last_sale']
    
    # Ciclo de vida de clientes
    customer_lifecycle = sales_df.groupby('customer_id')['periodo'].agg(['min', 'max']).reset_index()
    customer_lifecycle.columns = ['customer_id', 'customer_first_purchase', 'customer_last_purchase']
    
    # Ciclo de vida de combinaciones producto-cliente
    product_customer_lifecycle = sales_df.groupby(['product_id', 'customer_id'])['periodo'].agg(['min', 'max']).reset_index()
    product_customer_lifecycle.columns = ['product_id', 'customer_id', 'first_purchase', 'last_purchase']
    
    print(f"✅ Productos únicos: {len(product_lifecycle):,}")
    print(f"✅ Clientes únicos: {len(customer_lifecycle):,}")
    print(f"✅ Combinaciones producto-cliente: {len(product_customer_lifecycle):,}")
    
    return product_lifecycle, customer_lifecycle, product_customer_lifecycle

def calculate_metrics(y_true, y_pred, model_name="Modelo"):
    """
    Calcula métricas de evaluación
    """
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Métrica de error porcentual personalizada
    error_percentage = np.abs(y_pred - y_true).sum() / y_true.sum() * 100
    
    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'Error_Percentage': error_percentage
    }
    
    print(f"\n📊 MÉTRICAS DE {model_name.upper()}")
    print("-" * 40)
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    
    return metrics

def plot_predictions_vs_actual(y_true, y_pred, model_name="Modelo", sample_size=1000):
    """
    Grafica predicciones vs valores reales
    """
    # Tomar muestra si hay muchos datos
    if len(y_true) > sample_size:
        indices = np.random.choice(len(y_true), sample_size, replace=False)
        y_true_sample = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
        y_pred_sample = y_pred[indices]
    else:
        y_true_sample = y_true
        y_pred_sample = y_pred
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot
    ax1.scatter(y_true_sample, y_pred_sample, alpha=0.6, s=30)
    max_val = max(y_true_sample.max(), y_pred_sample.max())
    ax1.plot([0, max_val], [0, max_val], 'r--', linewidth=2)
    ax1.set_xlabel('Valores Reales')
    ax1.set_ylabel('Valores Predichos')
    ax1.set_title(f'{model_name} - Predicho vs Real')
    ax1.grid(True, alpha=0.3)
    
    # Histograma de errores
    errors = y_pred_sample - y_true_sample
    ax2.hist(errors, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax2.set_xlabel('Error de Predicción')
    ax2.set_ylabel('Frecuencia')
    ax2.set_title(f'{model_name} - Distribución de Errores')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✅ Funciones utilitarias definidas correctamente")

✅ Funciones utilitarias definidas correctamente


## Importamos datasets

In [21]:
# =============================================================================
# PARTE 3: LECTURA Y CARGA DE DATASETS
# =============================================================================

def load_datasets(data_path="../datasets/"):
    """
    Carga todos los datasets necesarios
    """
    print("📂 Iniciando carga de datasets...")
    
    try:
        # Cargar datos de ventas (sell-in)
        print("🔄 Cargando datos de ventas (sell-in.txt)...")
        sales = pd.read_csv(f"{data_path}sell-in.txt", 
                           sep="\t", 
                           dtype={"periodo": str})
        
        # Cargar datos de stock
        print("🔄 Cargando datos de stock (tb_stocks.txt)...")
        stocks = pd.read_csv(f"{data_path}tb_stocks.txt", 
                            sep="\t", 
                            dtype={"periodo": str})
        
        # Cargar información de productos
        print("🔄 Cargando información de productos (tb_productos.txt)...")
        product_info = pd.read_csv(f"{data_path}tb_productos.txt", sep="\t")
        
        # Cargar productos a predecir
        print("🔄 Cargando productos a predecir (product_id_apredecir201912.txt)...")
        products_to_predict = pd.read_csv(f'{data_path}product_id_apredecir201912.txt')
        
        print("✅ Todos los datasets cargados correctamente")
        
        return sales, stocks, product_info, products_to_predict
        
    except FileNotFoundError as e:
        print(f"❌ Error: No se pudo encontrar el archivo {e}")
        return None, None, None, None
    except Exception as e:
        print(f"❌ Error al cargar datasets: {e}")
        return None, None, None, None

def preprocess_datasets(sales, stocks, product_info, products_to_predict):
    """
    Preprocesa los datasets básicos
    """
    print("\n🔧 Iniciando preprocesamiento de datasets...")
    
    # Convertir fechas
    print("📅 Convirtiendo formatos de fecha...")
    validate_date_format(sales, 'periodo')
    validate_date_format(stocks, 'periodo')
    sales_raw=sales.copy()  # Guardar copia del dataset original
    # Mostrar información inicial
    display_dataframe_info(sales, "SALES (Original)")
    display_dataframe_info(stocks, "STOCKS")
    display_dataframe_info(product_info, "PRODUCT_INFO")
    
    # Verificar productos a predecir
    print(f"\n🎯 Productos a predecir: {len(products_to_predict):,}")
    if 'product_id' in products_to_predict.columns:
        predict_products_list = products_to_predict['product_id'].unique()
    else:
        # Si la columna tiene otro nombre, intentar la primera columna
        predict_products_list = products_to_predict.iloc[:, 0].unique()
    
    print(f"📋 Lista de productos a predecir (primeros 10): {predict_products_list[:10]}")
    
    # Validar que los productos a predecir estén en los datos históricos
    products_in_sales = sales['product_id'].unique()
    missing_products = set(predict_products_list) - set(products_in_sales)
    
    if missing_products:
        print(f"⚠️  Productos a predecir que NO están en ventas históricas: {len(missing_products)}")
        print(f"    Ejemplos: {list(missing_products)[:5]}")
    else:
        print("✅ Todos los productos a predecir tienen datos históricos")
    
    # Agregar información de productos a ventas
    print("🔗 Fusionando información de productos con ventas...")
    sales_with_info = sales.merge(product_info, on='product_id', how='left')
    
    # Verificar merge
    sales_without_info = sales_with_info[sales_with_info['cat1'].isnull()]
    if len(sales_without_info) > 0:
        print(f"⚠️  Ventas sin información de producto: {len(sales_without_info):,}")
        print(f"    Productos únicos sin info: {sales_without_info['product_id'].nunique()}")
    else:
        print("✅ Todas las ventas tienen información de producto")
    
    # Agregar stocks a ventas
    print("🔗 Fusionando información de stocks con ventas...")
    sales_complete = sales_with_info.merge(stocks, on=['periodo', 'product_id'], how='left')
    
    # Mostrar resultado final
    display_dataframe_info(sales_complete, "SALES (Completo)")
    
    return sales_complete, predict_products_list, sales_raw

# Ejecutar carga de datos
print("🚀 INICIANDO CARGA Y PREPROCESAMIENTO DE DATOS")
print("=" * 60)

# Cargar datasets
sales, stocks, product_info, products_to_predict = load_datasets()

if sales is not None:
    # Preprocesar datasets
    sales_complete, predict_products_list, sales_raw = preprocess_datasets(sales, stocks, product_info, products_to_predict)
    
    print(f"\n🎉 CARGA COMPLETADA EXITOSAMENTE")
    print(f"📊 Dataset final: {sales_complete.shape[0]:,} registros")
    print(f"🎯 Productos a predecir: {len(predict_products_list):,}")
    print(f"📅 Rango temporal: {sales_complete['periodo'].min()} a {sales_complete['periodo'].max()}")
else:
    print("❌ Error en la carga de datos. Verificar paths y archivos.")

🚀 INICIANDO CARGA Y PREPROCESAMIENTO DE DATOS
📂 Iniciando carga de datasets...
🔄 Cargando datos de ventas (sell-in.txt)...
🔄 Cargando datos de stock (tb_stocks.txt)...
🔄 Cargando información de productos (tb_productos.txt)...
🔄 Cargando productos a predecir (product_id_apredecir201912.txt)...
✅ Todos los datasets cargados correctamente

🔧 Iniciando preprocesamiento de datasets...
📅 Convirtiendo formatos de fecha...
🕐 Validando formato de fecha en columna 'periodo'...
✅ Fecha convertida desde formato YYYYMM
📅 Rango de fechas: 2017-01-01 00:00:00 a 2019-12-01 00:00:00
🕐 Validando formato de fecha en columna 'periodo'...
✅ Fecha convertida desde formato YYYYMM
📅 Rango de fechas: 2018-10-01 00:00:00 a 2019-12-01 00:00:00

📊 INFORMACIÓN DE SALES (ORIGINAL)
🔹 Forma: (2945818, 7)
🔹 Memoria utilizada: 157.32 MB
🔹 Rango de fechas: 2017-01-01 00:00:00 a 2019-12-01 00:00:00

📋 Tipos de datos:
int64             4
float64           2
datetime64[ns]    1
Name: count, dtype: int64

🔍 Valores nulos:
✅

In [28]:
# 📄 2. Cargar datasets
data = pd.read_csv("../datasets/sell-in.txt", sep="\t")
df_productos = pd.read_csv("../datasets/tb_productos.txt", sep="\t")

In [29]:
# 🧹 3. Preprocesamiento
# Convertir periodo a datetime
data['timestamp'] = pd.to_datetime(data['periodo'], format='%Y%m')

In [30]:


# Paso 1: Armado del Dataset
def prepare_dataset(df):
    return df.groupby(['product_id', 'periodo'])['tn'].sum().reset_index()

# Paso 2: Cálculo de la clase
def calculate_class(df):
    df['clase'] = df.groupby('product_id')['tn'].shift(-2)
    return df

# Paso 3: Feature Engineering
def feature_engineering(df):
    for i in range(1, 12):
        df[f'tn_{i}'] = df.groupby('product_id')['tn'].shift(i)
    return df

# Paso 4: Training Strategy
def prepare_training_data(df):
    magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021, 20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046, 20049,
               20051, 20052, 20053, 20055, 20008, 20001, 20017, 20086, 20180, 20193, 20320, 20532, 20612, 20637, 20807, 20838]
    
    training_data = df[(df['periodo'] == 201812) & (df['product_id'].isin(magicos))]
    training_data = training_data.dropna()
    print("Chequeamos la correcta dimension de datos de entrenamiento:")
    print(training_data.shape)
    print(training_data.columns)
    X = training_data[['tn'] + [f'tn_{i}' for i in range(1, 12)]]
    y = training_data['clase']
    
    return X, y

# Paso 5: Modelado
def train_model(X, y):
    model = LinearRegression()
    model.fit(X, y)
    return model

# Paso 6: Tratamiento de data faltante
def predict_and_fill(df, model):
    features = ['tn'] + [f'tn_{i}' for i in range(1, 12)]
    
    # Crear una copia del DataFrame para evitar advertencias
    results = df.copy()
    
    # Predicción para registros completos
    complete_mask = ~results[features].isnull().any(axis=1)
    results.loc[complete_mask, 'prediction'] = model.predict(results.loc[complete_mask, features])
    
    # Promedio para registros incompletos
    incomplete_mask = results[features].isnull().any(axis=1)
    results.loc[incomplete_mask, 'prediction'] = df['tn'].mean()
    
    return results

# Función principal
def main(df):
    # Paso 1
    df = prepare_dataset(df)
    
    # Paso 2
    df = calculate_class(df)
    
    # Paso 3
    df = feature_engineering(df)
    
    # Paso 4
    X, y = prepare_training_data(df)
    
    # Paso 5
    model = train_model(X, y)
    
    # Imprimir coeficientes
    coef_names = ['intercept'] + ['tn'] + [f'tn_{i}' for i in range(1, 12)]
    coef_values = [model.intercept_] + list(model.coef_)
    coef_df = pd.DataFrame({'coeficiente': coef_names, 'valor': coef_values})
    print(coef_df)
    
    # Paso 6
    results = predict_and_fill(df[df['periodo'] == 201912], model)
    
    return results

# Uso del script
# Asumiendo que tienes un DataFrame llamado 'data' con las columnas necesarias
# results = main(data)
# print(results)

In [32]:
results = main(data)
results

Chequeamos la correcta dimension de datos de entrenamiento:
(33, 15)
Index(['product_id', 'periodo', 'tn', 'clase', 'tn_1', 'tn_2', 'tn_3', 'tn_4',
       'tn_5', 'tn_6', 'tn_7', 'tn_8', 'tn_9', 'tn_10', 'tn_11'],
      dtype='object')
   coeficiente     valor
0    intercept  0.441467
1           tn -0.001339
2         tn_1  0.236558
3         tn_2  0.178208
4         tn_3 -0.060031
5         tn_4 -0.161875
6         tn_5 -0.007775
7         tn_6  0.151936
8         tn_7  0.043933
9         tn_8  0.142839
10        tn_9  0.103804
11       tn_10  0.119211
12       tn_11  0.073671


,product_id,periodo,tn,clase,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11,prediction
35,20001,201912,1504.68856,NaN,1397.37231,1561.50552,1660.00561,1261.34529,1678.99318,1109.93769,1629.78233,1647.63848,1470.65653,1259.09363,1275.77351,1162.707525
71,20002,201912,1087.30855,NaN,1423.57739,1979.53635,1090.18771,813.78215,1066.44999,928.36431,1034.98927,1287.62346,1083.62552,1043.01349,1266.78751,1183.640604
107,20003,201912,892.50129,NaN,948.29393,1081.36645,967.77116,635.59563,715.20314,662.38654,590.12515,565.33774,638.04010,758.32657,964.76919,684.763931
143,20004,201912,637.90002,NaN,723.94206,1064.69633,786.17140,482.13372,521.71519,667.19411,603.31081,466.70901,619.77084,441.70332,511.33713,580.484961
179,20005,201912,593.24443,NaN,606.91173,996.78275,879.52808,536.66800,745.74978,876.39696,897.26297,624.99880,488.21387,409.89950,363.58438,563.560780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31129,21265,201912,0.05007,NaN,0.06600,0.10921,0.01707,0.01593,0.02959,0.05121,0.17635,0.36405,0.01593,NaN,NaN,28.281626
31139,21266,201912,0.05121,NaN,0.06713,0.11831,0.02844,0.01480,0.05916,0.05235,0.17634,0.36178,0.01707,NaN,NaN,28.281626
31149,21267,201912,0.01569,NaN,0.04052,0.09676,0.01830,0.04054,0.07452,0.05882,0.24451,0.12291,0.21578,NaN,NaN,28.281626
31190,21271,201912,0.00298,NaN,0.01301,0.02453,0.02933,0.01784,0.01263,0.00445,0.04347,0.00185,0.00819,0.01041,0.00745,0.449656


In [63]:
# Load the sales data (tab-delimited)
sales = pd.read_csv(r"C:\Users\s1093678\OneDrive - Syngenta\Documents\Crop Protection\Cursos\Master en Data Science\21 - Labo III\repo\labo3-2025r\datasets\sell-in.txt", sep="\t", dtype={"periodo": str})

# Load the stocks data (tab-delimited)
stocks = pd.read_csv(r"C:\Users\s1093678\OneDrive - Syngenta\Documents\Crop Protection\Cursos\Master en Data Science\21 - Labo III\repo\labo3-2025r\datasets\tb_stocks.txt", sep="\t", dtype={"periodo": str})

# Load the product information data (tab-delimited)
product_info = pd.read_csv(r"C:\Users\s1093678\OneDrive - Syngenta\Documents\Crop Protection\Cursos\Master en Data Science\21 - Labo III\repo\labo3-2025r\datasets\tb_productos.txt", sep="\t")

# Load the product IDs to predict (tab-delimited)
products_to_predict = pd.read_csv(r'C:\Users\s1093678\OneDrive - Syngenta\Documents\Crop Protection\Cursos\Master en Data Science\21 - Labo III\repo\labo3-2025r\datasets\product_id_apredecir201912.txt')

In [64]:
sales['periodo'] = pd.to_datetime(sales['periodo'], format='%Y%m')
stocks['periodo'] = pd.to_datetime(stocks['periodo'], format='%Y%m')

In [65]:
# MONTHLY SALES

# Group by month and product, summing total sales in tons
monthly_sales = (
    sales.groupby(['periodo', 'product_id'])['tn']
    .sum()
    .reset_index()
)


In [66]:
# Merge stock_final into full_sales
data = sales.merge(stocks, on=['periodo', 'product_id'], how='left')

# Merge product info (static features)
data = data.merge(product_info, on='product_id', how='left')

In [67]:
import pandas as pd
import numpy as np

# Cargar el DataFrame original
#data = pd.read_csv('tu_archivo_original.csv')

# Cargar los productos a predecir


# Convertir la columna 'periodo' a datetime
monthly_sales['periodo'] = pd.to_datetime(monthly_sales['periodo'])

# Definir la fecha de corte (diciembre 2019)
fecha_corte = pd.Timestamp('2019-12-01')

# Función para calcular la media de los últimos n meses, excluyendo ceros
def calcular_media_sin_ceros(df, n_meses):
    fecha_inicio = fecha_corte - pd.DateOffset(months=n_meses)
    df_filtrado = df[(df['periodo'] >= fecha_inicio) & (df['periodo'] < fecha_corte)]
    
    # Agrupar por product_id y calcular la media, excluyendo ceros
    return df_filtrado[df_filtrado['tn'] > 0].groupby('product_id')['tn'].mean().reset_index()

# Calcular las medias para 18, 12, 6 y 3 meses
media_18 = calcular_media_sin_ceros(monthly_sales, 18)
media_12 = calcular_media_sin_ceros(monthly_sales, 12)
media_6 = calcular_media_sin_ceros(monthly_sales, 6)
media_3 = calcular_media_sin_ceros(monthly_sales, 3)

# Función para asegurar que todos los productos estén incluidos
def incluir_todos_productos(df):
    todos_productos = pd.merge(products_to_predict[['product_id']], df, on='product_id', how='left')
    todos_productos['tn'] = todos_productos['tn'].fillna(0)
    return todos_productos[['product_id', 'tn']]

# Aplicar la función a cada DataFrame de medias
todos_18 = incluir_todos_productos(media_18)
todos_12 = incluir_todos_productos(media_12)
todos_6 = incluir_todos_productos(media_6)
todos_3 = incluir_todos_productos(media_3)

# Guardar los resultados en CSV
todos_18.to_csv('prediccion_18_meses_sin_ceros.csv', index=False)
todos_12.to_csv('prediccion_12_meses_sin_ceros.csv', index=False)
todos_6.to_csv('prediccion_6_meses_sin_ceros.csv', index=False)
todos_3.to_csv('prediccion_3_meses_sin_ceros.csv', index=False)

# Mostrar información sobre los DataFrames
for df, nombre in zip([todos_18, todos_12, todos_6, todos_3], ['18 meses', '12 meses', '6 meses', '3 meses']):
    print(f"\nPredicción para {nombre}:")
    print(df.head())
    print(f"Número total de productos: {len(df)}")
    print(df.info())
    print(f"Número de productos con predicción > 0: {(df['tn'] > 0).sum()}")


Predicción para 18 meses:
   product_id           tn
0       20001  1522.657901
1       20002  1183.335757
2       20003   833.476484
3       20004   670.712084
4       20005   658.236986
Número total de productos: 780
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  780 non-null    int64  
 1   tn          780 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 12.3 KB
None
Número de productos con predicción > 0: 780

Predicción para 12 meses:
   product_id           tn
0       20001  1453.232564
1       20002  1168.949311
2       20003   774.753691
3       20004   622.854058
4       20005   649.885925
Número total de productos: 780
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   p